In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
import heapq  # ADDED: Data structure for efficient top-k queries

# Configure OS routines
import os
import subprocess # ADDED: Allows running system commands to auto-install missing packages
import sys # ADDED: For system-level operations

# ADDED: Auto-installs python-dotenv for secure secret management
subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv"])

# Initialize JupyterDash environment
JupyterDash.infer_jupyter_proxy_config()


# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# Custom CRUD  module file name and class name 
from aac_crud import AnimalShelter


#########################################################
# Algorithms & Data Structures: In-Memory Rescue Indexer
#########################################################
class RescueDogPriorityQueue:
    """ 
    Algorithmic Enhancement: Implements a Priority Queue (Max-Heap) 
    to rank dogs algorthmically based on suitability metrics.
    """
    def __init__(self):
        self.heap = []

    def calculate_suitability_score(self, dog, target_type):
        """
        Dynamic heuristic alorgithm calculating candiate fitness"""
        score = 0
        age = dog.get('age_upon_outcome_in_weeks', 0)

        if target_type == 'WR':  # Water Rescue optimal age range: 26-156 weeks
            score += 50 if (26.0<=age<=156.0) else 0
        elif target_type == 'DR': # Disaster Rescue optimal age range: 20-300 weeks
            score += 50 if (20.0<=age<=300.0) else 0

        # Prioritize intact animals for breeding/training programs
        if "Intact" in dog.get('sex_upon_outcome', ''):
            score += 25
        return score
        
    def process_and_sort_candidates(self, dog_list, rescue_type):
        """
        Process a list of dog candidates and return the top 10 based on suitability score.
        Ranks candidates using a heap structures in O(N log K) time complexity."""
        self.heap = []
        # Single pass to push items into the min-heap (inverted for max-heap behavior)
        for dog in dog_list:
            score = self.calculate_suitability_score(dog, rescue_type)
            # Use animal_id as a tie-breaker string to avoid heap comparison errors on dicts
            heapq.heappush(self.heap, (-score, dog.get('animal_id', ''), dog))

            sorted_list = []
            # Pop elements from highest to lowest score
            while self.heap:
                score,_, dog = heapq.heappop(self.heap)
                dog['suitability_score'] = -score  # Restore positive score value
                sorted_list.append(dog)
            return sorted_list   
         
############################
# Data Manipulation / Model
############################

# Retrieve sensitive credentials securely from the runtime environment

username = os.environ.get('MONGO_USER')
password = os.environ.get('MONGO_PASS')
host = 'localhost'
port = 27017
db = 'aac'
col = 'animals'

# Prevent application launch if credentials are missing
if not username or not password:
    print("CRITICAL SECURITY ERROR: Database credentials are not set in the environment variables!")
    print("Please set MONGO_USER and MONGO_PASS before running this script.")
    sys.exit(1)

# Connect to database via CRUD Module
shelter = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)

df.drop(columns=['_id'],inplace=True, errors='ignore') # ignore errors if column does not exist

## Debug
#print(len(df.to_dict(orient='records')))
#print(df.columns)

# Initialize the custom algorithm class
ranker = RescueDogPriorityQueue()

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    # Add logo and headers
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height':'10%', 'width':'10%'}),
    html.Center(html.B(html.H1('SNHU CS499: Dashbboard'))),
    html.Center(html.B(html.H2("Welcome Grazioso Salvare's Interactive Dashboard"))),
    html.Center(html.P("Enhanced by: Renee Cullen - 2026")),
    html.Hr(),
    dcc.RadioItems( # Code for filtering radio buttons
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'WR'},
            {'label': 'Mountain Rescue', 'value': 'MR'},
            {'label': 'Disaster Rescue', 'value': 'DR'},
            {'label': 'Reset', 'value': 'RESET'}
        ],
        value='RESET'
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
    # Set up features for your interactive data table to make it user-friendly for your client
                         editable = False,
                         filter_action = "native",
                         sort_action = "custom",  # Switched form native to custom to untilize the backend heapsort algorithm
                         sort_mode = "single",
                         column_selectable = "single",
                         row_selectable = "single",
                         row_deletable = False,
                         selected_columns = [],
                         selected_rows = [0],
                         page_action = "native",
                         page_current = 0,
                         page_size = 10,
                        
                        ),
    html.Br(),
    # This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
             style={'display' : 'flex'},
             children=[
                 dcc.Graph(
                     id='graph-id',
                     className='col s12 m6',
                 ),
                 html.Div(
                     id='map-id',
                     className='col s12 m6',
                 )
         ])
])


    #############################################
    # Interaction Between Components / Controller
    #############################################

@app.callback(
    [Output('datatable-id','data'),
     Output('datatable-id', 'columns')],
    [Input('filter-type', 'value')]
)

def update_dashboard(filter_type):
    query = {}
   
    
## Code to filter interactive data table with MongoDB queries

    if filter_type == 'WR': # Water Rescue
        query = {"animal_type": "Dog", "breed": {"$in":["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}}
        
    elif filter_type == 'MR': # Mountain Rescue
        query = {"animal_type": "Dog", "breed":{"$in":["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]}}

    elif filter_type == 'DR': # Disaster Rescue
        query = {"animal_type": "Dog", "breed":{"$in":["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]}}
                
    # Fetch data from crud module 
    data = shelter.read(query)
        
    # Remove the MongoDB ObjectId for JSON compatability
    if not data:
        return [], []
        
    for doc in data:
        doc.pop('_id', None)

    # CRITICAL ENHANCEMENT: Intercept data and pass it through the custom priority sorting algorithm
    if filter_type in  ['WR', 'DR', 'MR']:

        ranked_data = ranker.process_and_sort_candidates(data, filter_type)
    else:
        # Fallback for RESET to preserve performance on large raw data structures
        ranked_data = data

    # Fallback initialization if dataframe converts empty data structures
    if not ranked_data:
        return [], []

    df_ranked = pd.DataFrame.from_records(ranked_data)

    columns = [{"name": i, "id": i, "selectable": True} for i in df_ranked.columns]

    return ranked_data, columns

@app.callback(
    Output('graph-id', "figure"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    # Guard clause against empty states or uninitialized virtual views
    if not viewData:
        return px.pie(title="No Data Available to Render Chart")
    
    # Convert viewData to DataFrame
    df = pd.DataFrame.from_records(viewData)
    
    if df.empty:
        return px.pie(title="No Matching Breed Patterns Found")
    
    # Group all but the top 10 breeds into 'other'
    top_breeds = df['breed'].value_counts().nlargest(10).index
    df.loc[~df['breed'].isin(top_breeds), 'breed'] = 'other'
    
    # Create pie chart
    fig = px.pie(
        df,
        names='breed',
        title='Preferred Animal Breeds (Top 10)'
    )
    # Format for readability
    fig.update_layout(
        height=400,  # Make chart taller
        margin=dict(t=50, b=50, l=20, r=20) # Add space for title
    )
    fig.update_traces(
        textposition='inside', # Keep labels inside the slices
        textinfo='percent+label'
    )
    return fig

#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    # Handle initialization or empty data
    if not viewData or not index or len(index) == 0 :
        # Default view of Austin, TX if no row is selected
        return [
            dl.Map(style={'width': '100%', 'height': '450px'}, center=[30.2672, -97.7431], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]
    # Convert table data to Dataframe
    df= pd.DataFrame.from_records(viewData)
    
    # Get the row index (single selection)
    row_index = index[0]
    if row_index >= len(df):
        return []

    
    # Access coordinates using column nmaes to avoid index out-of-bounds errors
    # Use .get() to avoid KeyError if columns are missing
    lat = df.iloc[row_index].get('location_lat')
    lon = df.iloc[row_index].get('location_long')
    name = df.iloc[row_index].get('name', 'Animal')
    breed = df.iloc[row_index].get('breed', 'Uknown Breed')

    # Safeguard against missing or invalid coordinates
    if lat is None or lon is None:
        return []
    
    return [
        dl.Map(style={'width': '100%', 'height': '450px'}, center=[lat,lon], zoom=14, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(name),
                dl.Popup([
                    html.H1("Name"),
                    html.P(f"Breed: {breed}")
                ])
            ])
        ])
    ]
if __name__ == '__main__':
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
    app.run_server(mode='external', debug=True)